# 🎬 Autonomous Smart Video Editor (Shorts Generator)

**Single-file Colab version** – upload a long video, automatically detect silence and low-activity regions, keep the best segments, and download a polished short clip.

---
**Requirements:** FFmpeg (installed below), Python 3.8+, numpy, scipy

In [ ]:
# Install / verify dependencies
!apt-get install -y ffmpeg > /dev/null 2>&1
!pip install numpy scipy --quiet
print('✅ Dependencies ready')

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
CONFIG = {
    # Silence detection
    'silence_threshold_db': -40,   # dB – frames below this are considered silent
    'silence_min_duration': 1.5,   # seconds – minimum silence length to remove

    # Segment filtering
    'min_segment_score':    0.3,   # 0–1 – discard segments below this score
    'min_segment_duration': 2.0,   # seconds
    'max_segment_duration': 60.0,  # seconds
    'gap_fill_threshold':   0.5,   # seconds – merge segments closer than this

    # FFmpeg encoding
    'video_preset':  'fast',
    'video_crf':     23,
    'audio_bitrate': '128k',
    'threads':       4,
    'output_resolution': None,     # e.g. '1280x720' or None to keep original
}
print('✅ Config loaded')

In [ ]:
import json, os, struct, subprocess, tempfile, logging
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger('VideoEditor')


# ─── Data model ────────────────────────────────────────────────────────────────
@dataclass
class Segment:
    start: float
    end: float
    score: float = 0.0
    has_motion: bool = True
    tags: List[str] = field(default_factory=list)

    @property
    def duration(self) -> float:
        return max(0.0, self.end - self.start)


# ─── Video Processor ───────────────────────────────────────────────────────────
class VideoProcessor:
    def __init__(self, cfg: dict):
        self.cfg = cfg

    def analyze(self, video_path: str) -> List[Segment]:
        log.info('Analysing %s', video_path)
        meta = self._probe(video_path)
        duration = meta['duration']
        log.info('Duration: %.2f s', duration)

        audio_path = self._extract_audio(video_path)
        try:
            silence = self._detect_silence(audio_path, duration)
            activity = self._activity_scores(audio_path, duration)
        finally:
            if audio_path and os.path.exists(audio_path):
                os.remove(audio_path)

        return self._build_segments(duration, silence, activity)

    def _probe(self, path: str) -> dict:
        cmd = ['ffprobe', '-v', 'quiet', '-print_format', 'json',
               '-show_format', '-show_streams', path]
        out = subprocess.run(cmd, capture_output=True, check=True)
        data = json.loads(out.stdout)
        return {'duration': float(data['format'].get('duration', 0))}

    def _extract_audio(self, path: str) -> Optional[str]:
        tmp = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
        tmp.close()
        cmd = ['ffmpeg', '-y', '-i', path, '-vn', '-ac', '1', '-ar', '16000',
               '-f', 'wav', tmp.name]
        res = subprocess.run(cmd, capture_output=True)
        return tmp.name if res.returncode == 0 else None

    def _detect_silence(self, audio_path, duration) -> List[Tuple[float, float]]:
        if not audio_path:
            return [(0.0, duration)]
        th = self.cfg['silence_threshold_db']
        md = self.cfg['silence_min_duration']
        cmd = ['ffmpeg', '-i', audio_path,
               '-af', f'silencedetect=noise={th}dB:d={md}',
               '-f', 'null', '-']
        res = subprocess.run(cmd, capture_output=True)
        lines = res.stderr.decode('utf-8', errors='replace').splitlines()
        intervals, start = [], None
        for line in lines:
            if 'silence_start' in line:
                try: start = float(line.split('silence_start:')[1].split()[0])
                except: pass
            elif 'silence_end' in line and start is not None:
                try:
                    end = float(line.split('silence_end:')[1].split('|')[0].strip())
                    intervals.append((start, end))
                    start = None
                except: pass
        if start is not None:
            intervals.append((start, duration))
        log.info('Found %d silence interval(s)', len(intervals))
        return intervals

    def _activity_scores(self, audio_path, duration) -> List[float]:
        n = max(1, int(duration) + 1)
        scores = [0.0] * n
        if not audio_path:
            return scores
        try:
            samples = self._read_pcm(audio_path)
            sr = 16000
            for i in range(n):
                chunk = samples[i*sr:(i+1)*sr]
                if chunk:
                    rms = (sum(x*x for x in chunk) / len(chunk)) ** 0.5
                    scores[i] = min(1.0, rms / 3276.7)
        except Exception as e:
            log.warning('Activity estimation error: %s', e)
        return scores

    @staticmethod
    def _read_pcm(wav_path: str) -> List[int]:
        with open(wav_path, 'rb') as f:
            hdr = f.read(44)
            if len(hdr) < 44 or hdr[:4] != b'RIFF':
                return []
            ds = struct.unpack_from('<I', hdr, 40)[0]
            raw = f.read(ds)
        n = len(raw) // 2
        return list(struct.unpack(f'<{n}h', raw[:n*2]))

    def _build_segments(self, duration, silence, activity) -> List[Segment]:
        min_d = self.cfg['min_segment_duration']
        max_d = self.cfg['max_segment_duration']
        silent_sorted = sorted(silence)
        regions, prev = [], 0.0
        for ss, se in silent_sorted:
            if ss > prev:
                regions.append((prev, ss))
            prev = max(prev, se)
        if prev < duration:
            regions.append((prev, duration))

        segments = []
        for rs, re in regions:
            if re - rs < min_d:
                continue
            cur = rs
            while cur < re:
                end = min(cur + max_d, re)
                if end - cur < min_d:
                    break
                si, ei = int(cur), min(int(end) + 1, len(activity))
                chunk = activity[si:ei]
                score = sum(chunk) / max(1, len(chunk))
                segments.append(Segment(cur, end, score=score, has_motion=score > 0.1))
                cur = end
        return segments


# ─── Decision Engine ───────────────────────────────────────────────────────────
class DecisionEngine:
    W_AUDIO, W_MOTION, W_DUR = 0.5, 0.3, 0.2

    def __init__(self, cfg: dict):
        self.cfg = cfg

    def process(self, segments: List[Segment]) -> List[Segment]:
        scored = [self._score(s) for s in segments]
        filtered = [s for s in scored
                    if s.score >= self.cfg['min_segment_score']
                    and s.duration >= self.cfg['min_segment_duration']
                    and s.duration <= self.cfg['max_segment_duration']]
        merged = self._merge(filtered)
        log.info('Decision: %d → filtered %d → merged %d', len(scored), len(filtered), len(merged))
        return merged

    def _score(self, seg: Segment) -> Segment:
        max_d = self.cfg['max_segment_duration']
        seg.score = (
            seg.score * self.W_AUDIO
            + (1.0 if seg.has_motion else 0.0) * self.W_MOTION
            + min(1.0, seg.duration / max_d) * self.W_DUR
        )
        return seg

    def _merge(self, segs: List[Segment]) -> List[Segment]:
        if not segs:
            return []
        th = self.cfg['gap_fill_threshold']
        segs = sorted(segs, key=lambda s: s.start)
        out = [segs[0]]
        for s in segs[1:]:
            prev = out[-1]
            if s.start - prev.end <= th:
                prev.end = max(prev.end, s.end)
                prev.score = max(prev.score, s.score)
                prev.has_motion = prev.has_motion or s.has_motion
            else:
                out.append(s)
        return out


# ─── Renderer ──────────────────────────────────────────────────────────────────
class Renderer:
    def __init__(self, cfg: dict):
        self.cfg = cfg

    def render(self, src: str, segments: List[Segment], out: str) -> str:
        os.makedirs(os.path.dirname(out) or '.', exist_ok=True)
        with tempfile.TemporaryDirectory() as tmp:
            clips = self._cut(src, segments, tmp)
            if len(clips) == 1:
                self._encode(clips[0], out)
            else:
                lst = os.path.join(tmp, 'concat.txt')
                with open(lst, 'w') as f:
                    f.writelines(f"file '{c}'\n" for c in clips)
                self._concat_encode(lst, out)
        log.info('Output: %s', out)
        return out

    def _cut(self, src, segs, tmp) -> List[str]:
        paths = []
        for i, s in enumerate(segs):
            p = os.path.join(tmp, f'clip_{i:04d}.mp4')
            cmd = ['ffmpeg', '-y', '-ss', str(s.start), '-to', str(s.end),
                   '-i', src, '-c', 'copy', '-avoid_negative_ts', 'make_zero', p]
            subprocess.run(cmd, capture_output=True, check=True)
            paths.append(p)
        return paths

    def _encode_flags(self) -> List[str]:
        flags = ['-c:v', 'libx264', '-preset', self.cfg['video_preset'],
                 '-crf', str(self.cfg['video_crf']),
                 '-c:a', 'aac', '-b:a', self.cfg['audio_bitrate'],
                 '-threads', str(self.cfg['threads']),
                 '-movflags', '+faststart']
        if self.cfg.get('output_resolution'):
            flags += ['-vf', f"scale={self.cfg['output_resolution']}"]
        return flags

    def _encode(self, inp, out):
        subprocess.run(['ffmpeg', '-y', '-i', inp, *self._encode_flags(), out],
                       capture_output=True, check=True)

    def _concat_encode(self, lst, out):
        subprocess.run(['ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', lst,
                        *self._encode_flags(), out],
                       capture_output=True, check=True)


print('✅ Core classes loaded')

In [ ]:
from google.colab import files

print('📁 Select your video file to upload…')
uploaded = files.upload()
video_path = list(uploaded.keys())[0]
print(f'Uploaded: {video_path}')

In [ ]:
output_path = 'edited_output.mp4'

processor = VideoProcessor(CONFIG)
engine    = DecisionEngine(CONFIG)
renderer  = Renderer(CONFIG)

print('🔍 Step 1/3 – Analysing video…')
candidates = processor.analyze(video_path)
print(f'   Found {len(candidates)} candidate segments')

print('🧠 Step 2/3 – Scoring & filtering…')
final_segs = engine.process(candidates)
if not final_segs:
    raise RuntimeError('No suitable segments found. Try lowering min_segment_score.')
print(f'   Keeping {len(final_segs)} segments')
for s in final_segs:
    print(f'   [{s.start:.1f}s – {s.end:.1f}s]  score={s.score:.2f}')

print('🎬 Step 3/3 – Rendering output…')
renderer.render(video_path, final_segs, output_path)
print(f'\n✅ Done! Output saved to {output_path}')

In [ ]:
from google.colab import files
files.download(output_path)
print('📥 Download started!')